# 早停、排程、紀錄與自訂損失

對應原作業：Day 85–89。這是現代 PyTorch 練習；原始作業保留在 `../../homework/`。

從上到下執行即可。預設使用 CPU、合成資料，不需下載外部資料；改為真實資料時，先決定資料粒度與切分方式。

## 訓練控制
PyTorch 核心沒有把舊 Keras callback 一比一搬過來。每個 epoch 驗證後更新 scheduler、記錄類別別指標、保存最佳 `state_dict`，並依 patience 停止。

In [ ]:
import torch
from torch import nn
from pathlib import Path
from tempfile import TemporaryDirectory

torch.manual_seed(42)
model = nn.Linear(4, 2)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

best, wait, patience = float("inf"), 0, 3
with TemporaryDirectory() as directory:
    checkpoint = Path(directory) / "best.pt"
    for val_loss in [0.9, 0.7, 0.72, 0.74, 0.76]:  # 以實際驗證 loss 取代
        scheduler.step(val_loss)
        if val_loss < best:
            best, wait = val_loss, 0
            torch.save(model.state_dict(), checkpoint)
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
print("best validation loss:", best)


In [ ]:
def focal_mix_loss(logits: torch.Tensor, target: torch.Tensor, gamma: float = 2.0) -> torch.Tensor:
    # 0.3 × focal + 0.7 × cross entropy；兩項都保留梯度。
    ce = nn.functional.cross_entropy(logits, target, reduction="none")
    pt = torch.exp(-ce)
    return (0.3 * (1 - pt).pow(gamma) * ce + 0.7 * ce).mean()

X = torch.randn(8, 4)
y = torch.tensor([0, 1, 0, 1, 0, 0, 1, 1])
loss = focal_mix_loss(model(X), y)
loss.backward()
print("custom loss:", loss.item())

with torch.inference_mode():
    pred = model(X).argmax(dim=1)
    for cls in (0, 1):
        tp = ((pred == cls) & (y == cls)).sum().item()
        tn = ((pred != cls) & (y != cls)).sum().item()
        print(f"class {cls}: TP={tp}, TN={tn}")
